        # 📊 L01　看懂資料的長相
        **統計冒險之旅 2026**　｜　Day 1（09/05 六）🌄 統計之丘　｜　關卡　｜　🏅 100 XP

        📖 ISLP Ch2 觀念；資料：勇者咖啡八月銷售


        ### 🎯 這一關你會學到
        - 平均、中位數、標準差、四分位數與 describe()
- 直方圖、盒鬚圖、偏態與離群值（IQR 規則）
- groupby 分組摘要

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/rc/v1.1.0-rc.1/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "L01"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["1-1", "1-2", "1-3", "1-4", "1-5", "1-6"]
_XP_EACH = 16
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_1_1(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "平均"), 140.8037, 0.05): return (False, "平均 不對，用 df['金額'].mean()。")
    if not 約等於(抓變數(ns, "中位數"), 100.0, 0.5): return (False, "中位數 不對，用 .median()。")
    if not 約等於(抓變數(ns, "眾數"), 90, 0.5): return (False, "眾數 不對，用 .mode()[0]。")
    return (str(抓變數(ns, "最大的代表")).strip() == "平均", "想想看：右偏的資料，哪個代表會被大單拉高？")
任務定義("1-1", _check_1_1, 提示="右偏資料的平均會被大單拉高，所以三者中平均最大。")

def _check_1_2(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "標準差"), 93.4716, 0.1): return (False, "標準差 用 df['金額'].std()。")
    if not (約等於(抓變數(ns, "Q1"), 85.0, 0.5) and 約等於(抓變數(ns, "Q3"), 180.0, 0.5)): return (False, "Q1、Q3 用 quantile(0.25)、quantile(0.75)。")
    return (約等於(抓變數(ns, "IQR"), 95.0, 0.5), "IQR = Q3 - Q1。")
任務定義("1-2", _check_1_2, 提示="df['金額'].std()；quantile(0.25)；IQR = Q3 - Q1。")

def _check_1_3(run):
    out, ns = run()
    ok, msg = 資料框像(抓變數(ns, "摘要"), 種類="Series")
    if not ok: return (False, msg)
    if not 約等於(抓變數(ns, "第75百分位"), 180.0, 0.5): return (False, "第75百分位 = 摘要['75%']。")
    return (約等於(抓變數(ns, "最大值"), 1200.0, 0.5), "最大值 = 摘要['max']。")
任務定義("1-3", _check_1_3, 提示="摘要 是一個 Series，用 摘要['75%']、摘要['max'] 取值。")

def _check_1_4(run):
    out, ns = run()
    figs = run.figs
    if len(figs) < 2: return (False, "應該畫出兩張圖（兩個座標軸）。")
    t = " ".join(f["title"] + f["xlabel"] + f["ylabel"] for f in figs)
    if "金額" not in t: return (False, "直方圖的標題要包含「金額」。")
    if "時段" not in t: return (False, "盒鬚圖要用 時段 分組，標題包含「時段」。")
    return (True, "")
任務定義("1-4", _check_1_4, 提示="ax[0].set_title('八月每筆金額分布')；sns.boxplot(data=df, x='時段', y='金額', ax=ax[1])。")

def _check_1_5(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "上界"), 322.5, 0.5): return (False, "上界 = Q3 + 1.5 * (Q3 - Q1)。")
    if int(抓變數(ns, "離群筆數")) != 70: return (False, "離群筆數 = 金額 > 上界 的筆數。")
    return (約等於(抓變數(ns, "最大離群值"), 1200, 0.5), "最大離群值 = 離群['金額'].max()。")
任務定義("1-5", _check_1_5, 提示="上界 = Q3 + 1.5 * (Q3 - Q1)；len(離群)；離群['金額'].max()。")

def _check_1_6(run):
    out, ns = run()
    s = 抓變數(ns, "各分店平均")
    ok, msg = 資料框像(s, 列=3, 含欄位=["信義店", "板橋店", "中壢店"], 種類="Series")
    if not ok: return (False, msg)
    if not 約等於(s["板橋店"], 144.5, 0.15): return (False, "各分店平均 的數值不對，用 groupby('分店')['金額'].mean()。")
    if str(抓變數(ns, "最高分店")) != "板橋店": return (False, "最高分店 用 各分店平均.idxmax()。")
    c = 抓變數(ns, "各類別筆數")
    return (int(c["咖啡"]) == 1405, "各類別筆數 = df['類別'].value_counts()。")
任務定義("1-6", _check_1_6, 提示="groupby('分店')；.idxmax() 回傳最大值的索引；value_counts() 數類別次數。")

In [ ]:
import pandas as pd, numpy as np
import seaborn as sns, matplotlib.pyplot as plt
df = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.1.0-rc.1/data/coffee_sales_aug.csv")
df.head(3)

## 📊 1-1　集中趨勢：這群數字的「代表」是誰？
要用**一個數**代表八月每筆訂單的金額，有三種選法：

| 代表 | 怎麼算 | 什麼時候用 |
|---|---|---|
| **平均數 mean** | 全部加起來除以筆數 | 資料大致對稱、沒有極端值 |
| **中位數 median** | 排好隊，站在正中間的那個人 | 有極端值時（薪資、房價、金額） |
| **眾數 mode** | 出現最多次的值 | 類別資料、想知道「最常見」 |

> 🍜 **億萬富翁進小吃店**：小吃店裡 10 個客人平均月薪 5 萬，億萬富翁一走進來，平均變成 900 萬——但中位數幾乎不動。平均會被極端值拉走，中位數不會。

In [ ]:
金額 = df["金額"]
print("平均", round(金額.mean(), 1), "| 中位數", 金額.median(), "| 眾數", 金額.mode()[0])
print("平均 > 中位數 → 右邊有一群大單把平均拉高了")

## 1-2　離散程度：大家離「代表」多遠？
只知道代表還不夠，還要知道**散多開**。

| 指標 | 一句話 |
|---|---|
| **變異數 var** | 每個值離平均的距離平方，取平均 |
| **標準差 std** | 變異數開根號 → 單位和原資料一樣（元），最常用 |
| **四分位數 Q1／Q3** | 排隊後 25%、75% 位置的人 |
| **IQR = Q3 − Q1** | 中間一半的人散多開，不受極端值影響 |

> 🌡️ **量體溫**：體溫 37 度上下 0.5 度是正常範圍。標準差就是「這群人的正常範圍有多寬」。

In [ ]:
print("標準差", round(金額.std(), 1), "| 變異數", round(金額.var(), 1))
q1, q3 = 金額.quantile(0.25), 金額.quantile(0.75)
print("Q1", q1, "| Q3", q3, "| IQR", q3 - q1)
金額.describe()          # 一次看完：筆數、平均、標準差、最小、Q1、中位數、Q3、最大

## 1-3　資料的長相：直方圖與盒鬚圖
數字看不出「形狀」，畫圖才看得出來。
- **直方圖 histogram**：把金額切成一格一格，數每格有幾筆 → 看出偏態（skew）：尾巴往右拖＝右偏。
- **盒鬚圖 boxplot**：📦 一箱資料的開箱照——箱子是中間一半（Q1～Q3），箱中線是中位數，鬍鬚是正常範圍，外面的點是**離群值**。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["金額"], bins=30, ax=ax[0]); ax[0].set_title("八月每筆金額的直方圖（右偏）")
sns.boxplot(data=df, x="分店", y="金額", ax=ax[1]); ax[1].set_title("各分店金額的盒鬚圖")
plt.tight_layout(); plt.show()

## 1-4　離群值：IQR 規則
盒鬚圖上那些「外面的點」怎麼決定的？最常用的規則：**大於 Q3 + 1.5×IQR 或小於 Q1 − 1.5×IQR** 就算離群值。
離群值不一定是錯的（八月的團體訂單是真的），但要**知道它們在**，因為平均會被拉走、模型會被帶偏。

In [ ]:
iqr = q3 - q1
上界 = q3 + 1.5 * iqr
print("上界", 上界, "| 超過上界的筆數", (金額 > 上界).sum())
df[金額 > 上界].sort_values("金額", ascending=False).head(5)

## 1-5　分組摘要：不同分店、類別、時段長得一樣嗎？
第一部學過的 `groupby` 在統計課會一直用：**一組一組算平均、中位數、標準差**，就是最基本的比較。

In [ ]:
print(df.groupby("分店")["金額"].agg(["count", "mean", "median", "std"]).round(1))
print(df["類別"].value_counts())                    # 類別資料用「次數」描述
print(df.groupby("時段")["金額"].mean().round(1))

### 🎯 任務 1-1　三種代表

算出八月金額的 `平均`、`中位數`、`眾數`（`mode()[0]`），並把三者中**最大的那個代表的名字**（字串 `"平均"`、`"中位數"` 或 `"眾數"`）存成 `最大的代表`。

In [ ]:
# 🎯 任務 1-1　三種代表（請保留這一行）
平均 = df["金額"].mean()
中位數 = ???
眾數 = ???
最大的代表 = ???
print(round(平均, 1), 中位數, 眾數, 最大的代表)

In [ ]:
檢查("1-1")   # ◀ 執行這一格，看看任務 1-1 有沒有過關

### 🎯 任務 1-2　離散程度

算出金額的 `標準差`（pandas 預設的 `.std()`）、`Q1`、`Q3`（`quantile(0.25)`、`quantile(0.75)`）與 `IQR`。

In [ ]:
# 🎯 任務 1-2　離散程度（請保留這一行）
標準差 = ???
Q1 = ???
Q3 = ???
IQR = ???
print(round(標準差, 1), Q1, Q3, IQR)

In [ ]:
檢查("1-2")   # ◀ 執行這一格，看看任務 1-2 有沒有過關

### 🎯 任務 1-3　看懂 describe()

把 `df['金額'].describe()` 存成 `摘要`，再從 `摘要` 取出 `第75百分位`（`摘要['75%']`）與 `最大值`（`摘要['max']`）。

In [ ]:
# 🎯 任務 1-3　看懂 describe()（請保留這一行）
摘要 = df["金額"].describe()
第75百分位 = ???
最大值 = ???
print(摘要)
print(第75百分位, 最大值)

In [ ]:
檢查("1-3")   # ◀ 執行這一格，看看任務 1-3 有沒有過關

### 🎯 任務 1-4　畫出資料的長相

畫兩張圖：(1) 金額的**直方圖**（`sns.histplot`，`bins=30`），標題要包含「金額」；(2) 以 `時段` 分組的金額**盒鬚圖**（`sns.boxplot(data=df, x='時段', y='金額')`），標題要包含「時段」。

In [ ]:
# 🎯 任務 1-4　畫出資料的長相（請保留這一行）
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["金額"], bins=30, ax=ax[0]); ax[0].set_title(???)
sns.boxplot(data=df, x=???, y="金額", ax=ax[1]); ax[1].set_title(???)
plt.tight_layout(); plt.show()

In [ ]:
檢查("1-4")   # ◀ 執行這一格，看看任務 1-4 有沒有過關

### 🎯 任務 1-5　IQR 規則抓離群值

用 IQR 規則算出 `上界`（Q3 + 1.5×IQR）、`離群筆數`（金額大於上界的筆數），並把離群值中**最大的一筆金額**存成 `最大離群值`。

In [ ]:
# 🎯 任務 1-5　IQR 規則抓離群值（請保留這一行）
Q1, Q3 = df["金額"].quantile(0.25), df["金額"].quantile(0.75)
上界 = ???
離群 = df[df["金額"] > 上界]
離群筆數 = ???
最大離群值 = ???
print(上界, 離群筆數, 最大離群值)

In [ ]:
檢查("1-5")   # ◀ 執行這一格，看看任務 1-5 有沒有過關

### 🎯 任務 1-6　分組摘要

算出 `各分店平均`（各分店金額平均，四捨五入 1 位，Series）、`最高分店`（平均最高的分店名稱，`idxmax()`），以及 `各類別筆數`（`value_counts()`）。

In [ ]:
# 🎯 任務 1-6　分組摘要（請保留這一行）
各分店平均 = df.groupby(???)["金額"].mean().round(1)
最高分店 = ???
各類別筆數 = ???
print(各分店平均)
print("平均最高：", 最高分店)
print(各類別筆數)

In [ ]:
檢查("1-6")   # ◀ 執行這一格，看看任務 1-6 有沒有過關

## 🌟 進階挑戰（不計分）
1. 用 `df.groupby(['分店', '時段'])['金額'].mean().unstack()` 做交叉表，哪個分店的晚上生意最好？
2. 把金額取 `np.log()` 再畫直方圖，右偏會怎麼變？（很多模型喜歡「取 log 後比較對稱」的資料）

---
## 🔑 通關密語
　你已經能用數字和圖「看懂」一份資料了。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🎲 L02 機率與分布** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.1.0-rc.1/notebooks/L02_distributions.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/rc/v1.1.0-rc.1/